In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
import os
import shutil
import subprocess
import sys
import time

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    _t0 = time.time()

    def _el():
        return f"{time.time() - _t0:6.1f}s"

    # Write a placeholder submission before anything risky runs. rules.md:
    # "it's auto-generated as long as the agent acts on the games" -- which
    # means if this cell's own setup throws *before* main.py ever plays a
    # single game (nothing below this point is guarded the way choose_action/
    # is_done now are), no submission file is ever produced at all. main.py's
    # own harness overwrites this placeholder with the real recorded
    # submission as soon as the agent acts on at least one game, so this is
    # pure insurance against a total setup failure, not a replacement for
    # real play.
    import pandas as pd
    pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet("/kaggle/working/submission.parquet", index=False)
    print(f"[{_el()}] === step: wrote placeholder submission.parquet ===", flush=True)

    # Kaggle's own guidance: a code-competition rerun errors out if the agent
    # doesn't make its first move (open the environments) within roughly 15
    # minutes of container start. The original version of this cell spent up
    # to 10 of those minutes purely blocked on the gateway-readiness curl
    # check below (retry-max-time=600) *before* any of our own setup even
    # started -- on a slow/contended host that could plausibly eat most of
    # the budget by itself. Two changes here: (1) shrink the retry window
    # to 90s -- if the gateway isn't reachable in 90s it's very unlikely to
    # become reachable in the time remaining anyway, and this used to be
    # silent dead time; (2) run it as a background subprocess so our own
    # file-copying/import setup (which doesn't depend on the gateway at
    # all) happens *concurrently* with the wait instead of strictly after
    # it, then join on it right before main.py needs the gateway to be up.
    print(f"[{_el()}] === step: starting gateway wait in background ===", flush=True)
    gateway_proc = subprocess.Popen(
        ["curl", "--fail", "--retry", "999", "--retry-all-errors",
         "--retry-delay", "2", "--retry-max-time", "90",
         "http://gateway:8001/api/games"],
    )

    print(f"[{_el()}] === step: checking torch/numpy availability ===", flush=True)
    # Internet access is disabled during scored runs (competition rule),
    # so the pip-install fallback below can only ever help in a manual
    # test push with enable_internet=true -- it cannot rescue a real
    # scored run if torch/numpy are genuinely missing there. Kept anyway
    # so a real scored-run failure here produces this exact clear message
    # instead of a silent crash; there's no actual fallback available if
    # it triggers for real, only a better error message. Expectation is
    # that both are already present in Kaggle's standard notebook image.
    try:
        import torch  # noqa: F401
        import numpy  # noqa: F401
        print(f"[{_el()}] torch {torch.__version__}, numpy {numpy.__version__} already available", flush=True)
    except ImportError as e:
        print(f"[{_el()}] torch/numpy not available ({e}) -- no internet during scored runs, "
              f"so there is no fallback; attempting pip install anyway in case this "
              f"is a manual test push with internet enabled", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "torch", "numpy"], check=True)
        import torch  # noqa: F401
        print(f"[{_el()}] installed torch {torch.__version__}", flush=True)

    print(f"[{_el()}] === step: copying competition harness repo (excluding .git) ===", flush=True)
    # A plain `cp -r` would also copy the harness repo's .git history if
    # present in the competition dataset (CLAUDE.md's own gotcha: the
    # Kaggle dataset zip mirrors ARC-AGI-3-Agents including .git) -- never
    # needed at runtime and a plausible source of avoidable copy time.
    # shutil.copytree + ignore_patterns skips it without depending on
    # rsync being present in the image.
    shutil.copytree(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents",
        "/kaggle/working/ARC-AGI-3-Agents",
        ignore=shutil.ignore_patterns(".git"),
    )

    print(f"[{_el()}] === step: copying jepa package + checkpoints + agent ===", flush=True)
    # hypothesis_agent.py's own path resolution
    # (Path(__file__).resolve().parents[3]) expects jepa/ and checkpoints/
    # to sit directly under the repo root -- here, /kaggle/working -- one
    # level above ARC-AGI-3-Agents, matching this project's own local
    # layout (ARC-AGI-3-Agents/agents/templates/hypothesis_agent.py is
    # 3 parents deep from the repo root in both places).
    #
    # ROOT CAUSE of every prior scored-run failure (found via a free,
    # non-rerun diagnostic push that unconditionally walked /kaggle/input):
    # a dataset_sources attachment mounts at
    # /kaggle/input/datasets/<owner>/<slug>/, NOT /kaggle/input/<slug>/ --
    # the competition_sources path (/kaggle/input/competitions/<comp>/,
    # used correctly above) already followed this nested convention, but
    # every version of this cell used the un-nested form for our own
    # dataset. Every shutil.copytree/copy below was hitting a
    # FileNotFoundError on the very first call, before main.py ever ran a
    # single game -- explaining why every other fix (timing, GPU,
    # heartbeat try/except, placeholder-submission-first) never had a
    # chance to matter: cell 1 itself was crashing before any of that code
    # ran. Confirmed directly: torch 2.10.0 (Kaggle's shipped version, vs
    # our local torch==2.12.1) loads all three checkpoints without issue
    # once the path is correct -- the version mismatch was a red herring.
    shutil.copytree("/kaggle/input/datasets/calamitychasm/jepa-hypothesis-agent/jepa", "/kaggle/working/jepa")
    shutil.copytree("/kaggle/input/datasets/calamitychasm/jepa-hypothesis-agent/checkpoints", "/kaggle/working/checkpoints")
    shutil.copy(
        "/kaggle/input/datasets/calamitychasm/jepa-hypothesis-agent/hypothesis_agent.py",
        "/kaggle/working/ARC-AGI-3-Agents/agents/templates/hypothesis_agent.py",
    )

    print(f"[{_el()}] === step: verifying copied files exist ===", flush=True)
    for p in [
        "/kaggle/working/jepa/models/moe_predictor.py",
        "/kaggle/working/checkpoints/encoder_moe.pt",
        "/kaggle/working/checkpoints/moe_predictor.pt",
        "/kaggle/working/checkpoints/value_head.pt",
        "/kaggle/working/checkpoints/game_vocab_moe.json",
        "/kaggle/working/ARC-AGI-3-Agents/agents/templates/hypothesis_agent.py",
    ]:
        assert os.path.exists(p), f"missing expected file: {p}"
    print(f"[{_el()}] all expected files present", flush=True)

    print(f"[{_el()}] === step: writing agents/__init__.py ===", flush=True)
    # Minimal __init__.py that only imports what we need (the original
    # eagerly imports templates with unmet deps like langgraph, smolagents)
    # -- registers our agent as "hypothesis".
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write(
            "from typing import Type, cast\n"
            "from dotenv import load_dotenv\n"
            "from .agent import Agent, Playback\n"
            "from .swarm import Swarm\n"
            "from .templates.random_agent import Random\n"
            "from .templates.hypothesis_agent import Hypothesis\n"
            "\n"
            "load_dotenv()\n"
            "\n"
            "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
            "    \"random\": Random,\n"
            "    \"hypothesis\": Hypothesis,\n"
            "}\n"
        )

    print(f"[{_el()}] === step: sanity-importing our agent before running main.py ===", flush=True)
    # Fail here, with a real traceback, rather than have an import error
    # deep inside main.py's own agent-loading path look identical to any
    # other failure.
    sys.path.insert(0, "/kaggle/working/ARC-AGI-3-Agents")
    sys.path.insert(0, "/kaggle/working")
    from agents.templates.hypothesis_agent import Hypothesis  # noqa: F401
    print(f"[{_el()}] agent import OK", flush=True)

    print(f"[{_el()}] === step: writing .env ===", flush=True)
    # Overrides .env.example defaults; main.py loads this second with
    # override=True, so it wins.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write(
            "SCHEME=http\n"
            "HOST=gateway\n"
            "PORT=8001\n"
            "ARC_API_KEY=test-key-123\n"
            "ARC_BASE_URL=http://gateway:8001/\n"
            "OPERATION_MODE=online\n"
            "ENVIRONMENTS_DIR=\n"
            "RECORDINGS_DIR=/kaggle/working/server_recording\n"
        )

    print(f"[{_el()}] === step: joining background gateway-wait ===", flush=True)
    # main.py's own /api/games check has only a 10s timeout and no retry
    # loop of its own (confirmed by reading main.py directly) -- it relies
    # entirely on the gateway already being reachable by the time it runs.
    # All of our own setup above ran concurrently with this wait rather
    # than after it.
    gw_rc = gateway_proc.wait()
    print(f"[{_el()}] gateway curl exited with code {gw_rc}", flush=True)
    if gw_rc != 0:
        print(f"[{_el()}] WARNING: gateway did not respond within the shortened retry "
              f"window -- proceeding to main.py anyway (it may still come up moments "
              f"later); if this run errors, this line is the first thing to check.",
              flush=True)

    print(f"[{_el()}] === step: running agent ===", flush=True)
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", "hypothesis"],
        cwd="/kaggle/working/ARC-AGI-3-Agents",
        env={**os.environ, "MPLBACKEND": "agg"},
    )
    print(f"[{_el()}] === main.py exited with code {result.returncode} ===", flush=True)


In [ ]:
# Non-rerun mode: produce a dummy submission
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)